# PrivateLocalAgent

1. Kernel → Restart Kernel  
2. 运行单元格 1，等到 `ready`  
3. 运行单元格 2，开始对话

In [ ]:
import os, sys, subprocess
from pathlib import Path

def log(msg):
    print(msg, flush=True)

def find_root() -> Path:
    candidates = [
        Path("/workspace/Radeon-hackathon-2026-07"),
        Path.cwd(),
        Path.cwd().parent,
    ]
    here = Path.cwd()
    for p in [here, *here.parents]:
        candidates.append(p)
    for root in candidates:
        if (root / "src" / "config.py").is_file() and (root / "notebooks").is_dir():
            return root.resolve()
    raise FileNotFoundError("找不到项目根目录（需含 src/config.py）")

ROOT = find_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
log(f"ROOT={ROOT}")

persist = Path("/workspace/persistence")
if persist.is_dir():
    os.environ.setdefault("PLA_DATA_ROOT", str(persist / "PrivateLocalAgent"))
    os.environ.setdefault("HF_HOME", str(persist / "huggingface"))
    Path(os.environ["PLA_DATA_ROOT"]).mkdir(parents=True, exist_ok=True)
    Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("USE_ROCM_AITER_ROPE_BACKEND", "0")

log("[0] pip...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "chromadb", "sentence-transformers", "pypdf", "pyyaml",
    "python-dotenv", "pydantic", "openai", "ipywidgets",
    "transformers", "accelerate", "safetensors", "sentencepiece",
    "Pillow", "rapidocr-onnxruntime",
])

from src.agent.agent import PrivateAgent
from src.agent.multi_agent import MultiAgentOrchestrator
from src.agent.tools import ToolRegistry
from src.apps.judge_script import ensure_judge_ocr_image
from src.config import load_settings
from src.llm.backend import build_llm
from src.memory.memory import SessionMemory
from src.privacy.audit import AuditTrail
from src.rag.store import VectorStore
from src.skills import SkillRegistry

settings = load_settings()
upload_dir = settings.resolve(settings.paths.upload_dir)
ensure_judge_ocr_image(upload_dir)

log("[1] KB...")
try:
    store = VectorStore(settings)
except Exception as exc:
    log(f"KB 加载失败（检查 HF_ENDPOINT / HF_HOME）: {exc}")
    raise
n = store.ensure_sample_docs(settings.resolve(settings.paths.sample_docs))
log(f"    ingested={n} chunks={store.count()}")

memory = SessionMemory(settings.resolve(settings.agent.memory_path))
skills = SkillRegistry(settings.resolve(settings.paths.generated_projects))
tools = ToolRegistry(store, memory, upload_dir, skill_registry=skills)
audit = AuditTrail(settings.resolve("data/memory/audit.jsonl"))

log("[2] load LLM...")
try:
    llm = build_llm(settings.llm)
except Exception as exc:
    log(f"LLM 加载失败（检查 ROCm / 模型缓存）: {exc}")
    raise
agent = PrivateAgent(llm, tools, memory, settings.agent.max_steps, audit=audit)
orch = MultiAgentOrchestrator(agent, tools)
log("ready")

In [ ]:
from src.app.notebook_visual import launch_notebook_visual

assert "orch" in globals(), "请先运行上一单元格直到出现 ready"
ui = launch_notebook_visual(orch, default_mode="chat")